# Lista MMQ — Mínimos Quadrados e Integração Numérica

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)

def r2(y, yhat):
    ss_res = np.sum((y-yhat)**2); ss_tot = np.sum((y-np.mean(y))**2)
    return 1 - ss_res/ss_tot if ss_tot else 1.0

def trapz(f, a, b, n):
    x = np.linspace(a, b, n+1); y = f(x)
    return (b-a)/(2*n) * (y[0] + 2*np.sum(y[1:-1]) + y[-1])

def simp13(f, a, b, n):
    if n%2: n+=1
    x = np.linspace(a, b, n+1); y = f(x)
    return (b-a)/(3*n) * (y[0] + 4*np.sum(y[1::2]) + 2*np.sum(y[2:-1:2]) + y[-1])

def gauss_quad(f, a, b, n):
    xi, wi = np.polynomial.legendre.leggauss(n)
    t = 0.5*(b-a)*xi + 0.5*(b+a)
    return 0.5*(b-a)*np.sum(wi*f(t))


## Questão 1
Ajuste de V(i): modelos linear, logarítmico e quadrático.

In [2]:
i = np.array([0.25, 0.75, 1.25, 1.50, 2.00])
V = np.array([-0.45,-0.60, 0.70, 1.88, 6.00])

# Linear: V = a0 + a1*i
A = np.column_stack([np.ones_like(i), i])
a0, a1 = np.linalg.lstsq(A, V, rcond=None)[0]
V_lin = a0 + a1*i
print(f"Linear:     V = {a0:.4f} + {a1:.4f}*i   R²={r2(V,V_lin):.4f}  V(1.15)={a0+a1*1.15:.4f}")

# Logarítmico: V = a + b*ln(i)
A = np.column_stack([np.ones_like(i), np.log(i)])
a, b = np.linalg.lstsq(A, V, rcond=None)[0]
V_log = a + b*np.log(i)
print(f"Logarítm.:  V = {a:.4f} + {b:.4f}*ln(i)  R²={r2(V,V_log):.4f}  V(1.15)={a+b*np.log(1.15):.4f}")

# Quadrático: V = a0 + a1*i + a2*i²
c = np.polyfit(i, V, 2)
V_quad = np.polyval(c, i)
print(f"Quadrát.:   V = {c[0]:.4f}i² + {c[1]:.4f}i + {c[2]:.4f}  R²={r2(V,V_quad):.4f}  V(1.15)={np.polyval(c,1.15):.4f}")

print("Nota: exp. e potência não aplicáveis (V≤0 na amostra).")


Linear:     V = -2.5729 + 3.5468*i   R²=0.7850  V(1.15)=1.5060
Logarítm.:  V = 1.6747 + 2.3946*ln(i)  R²=0.5244  V(1.15)=2.0094
Quadrát.:   V = 3.3945i² + -4.0093i + 0.3886  R²=0.9988  V(1.15)=0.2670
Nota: exp. e potência não aplicáveis (V≤0 na amostra).


## Questão 2
Ajuste linear e quadrático para dispersão de dados.

In [3]:
x = np.array([6,7,11,15,17,21,23,29,33,37,39], dtype=float)
y = np.array([29,21,29,14,21,15,7,7,13,0,3], dtype=float)

A = np.column_stack([np.ones_like(x), x])
a0, a1 = np.linalg.lstsq(A, y, rcond=None)[0]
y_lin = a0 + a1*x
print(f"Linear:    y = {a0:.4f} + {a1:.4f}x   R²={r2(y,y_lin):.4f}")

c = np.polyfit(x, y, 2)
y_quad = np.polyval(c, x)
print(f"Quadrát.:  y = {c[0]:.4f}x² + {c[1]:.4f}x + {c[2]:.4f}   R²={r2(y,y_quad):.4f}")


Linear:    y = 30.4874 + -0.7410x   R²=0.7760
Quadrát.:  y = 0.0075x² + -1.0782x + 33.3357   R²=0.7828


## Questão 3
Modelos linear, potência e quadrático para dados técnicos.

In [4]:
x = np.array([5,10,15,20,25,30,35,40,45,50], dtype=float)
y = np.array([17,24,31,33,37,37,40,40,42,41], dtype=float)

A = np.column_stack([np.ones_like(x), x])
a0, a1 = np.linalg.lstsq(A, y, rcond=None)[0]
print(f"Linear:    y = {a0:.4f} + {a1:.4f}x   R²={r2(y, a0+a1*x):.4f}")

A = np.column_stack([np.ones_like(np.log(x)), np.log(x)])
lnb, m = np.linalg.lstsq(A, np.log(y), rcond=None)[0]
b = np.exp(lnb)
print(f"Potência:  y = {b:.4f}*x^{m:.4f}   R²={r2(y, b*x**m):.4f}")

c = np.polyfit(x, y, 2)
print(f"Quadrát.:  R²={r2(y, np.polyval(c,x)):.4f}")


Linear:    y = 20.6000 + 0.4945x   R²=0.8385
Potência:  y = 9.9529*x^0.3851   R²=0.9377
Quadrát.:  R²=0.9800


## Questão 4
Ajuste para dados de túnel de vento: log, potência e exponencial.

In [5]:
v = np.array([10,20,30,40,50,60,70,80], dtype=float)
F = np.array([25,70,380,550,610,1220,830,1450], dtype=float)

A = np.column_stack([np.ones_like(np.log(v)), np.log(v)])
a, b_l = np.linalg.lstsq(A, np.log(F), rcond=None)[0]
print(f"Potência:  F = e^{a:.4f} * v^{b_l:.4f}  R²={r2(F, np.exp(a)*v**b_l):.4f}")

A = np.column_stack([np.ones_like(v), np.log(v)])
a0, a1 = np.linalg.lstsq(A, F, rcond=None)[0]
print(f"Logarít.:  F = {a0:.4f} + {a1:.4f}*ln(v)  R²={r2(F, a0+a1*np.log(v)):.4f}")

A = np.column_stack([np.ones_like(v), v])
a0, a1 = np.linalg.lstsq(A, np.log(F), rcond=None)[0]
print(f"Exp.:      F = e^{a0:.4f} * e^{a1:.4f}v  R²={r2(F, np.exp(a0+a1*v)):.4f}")


Potência:  F = e^-1.2941 * v^1.9842  R²=0.8088
Logarít.:  F = -1683.3753 + 640.8896*ln(v)  R²=0.7867
Exp.:      F = e^3.5267 * e^0.0528v  R²=0.2373


## Questão 5
Regressão múltipla: y = a0 + a1*x1 + a2*x2.

In [6]:
x1 = np.array([0,1,1,2,2,3,3,4,4], dtype=float)
x2 = np.array([0,1,2,1,2,1,2,1,2], dtype=float)
y  = np.array([15.1,17.9,12.7,25.6,20.5,35.1,29.7,45.4,40.2])

A = np.column_stack([np.ones_like(x1), x1, x2])
a0, a1, a2 = np.linalg.lstsq(A, y, rcond=None)[0]
yhat = a0 + a1*x1 + a2*x2
print(f"a0={a0:.4f}  a1={a1:.4f}  a2={a2:.4f}  R²={r2(y,yhat):.4f}")


a0=14.4609  a1=9.0252  a2=-5.7043  R²=0.9955


## Questão 6
Circuito RL — energia WR por integração numérica (Trapézio, Simpson 1/3 e 3/8).

In [7]:
# i(t) = e^(-2t)*(A*sin(5t) + B*cos(5t) + C*e^(-998t))
# Coeficientes da solução exata: i'+1000i = 100*e^(-2t)*sin(5t)
# Resolvendo analiticamente: A=99800/996029, B=-500/996029, C=500/996029
A_c = 99800/996029; B_c = -500/996029; C_c = 500/996029
i_t = lambda t: np.exp(-2*t)*(A_c*np.sin(5*t) + B_c*np.cos(5*t)) + C_c*np.exp(-1000*t)
R = 100.0
g = lambda t: R * i_t(t)**2

# Referência simbólica ≈ 0.10770 J
W_ref = 0.10770

print(f"{'n':>5}  {'Trapézio':>14}  {'Simp 1/3':>14}  {'Simp 3/8':>14}")
for n in [6, 12, 24]:
    tr = trapz(g, 0, 2, n)
    s1 = simp13(g, 0, 2, n)
    # Simpson 3/8: n deve ser múltiplo de 3
    n3 = n if n%3==0 else n+3-n%3
    x3 = np.linspace(0, 2, n3+1); y3 = g(x3); h3 = 2/n3
    s3 = 3*h3/8*(y3[0]+y3[-1] + 3*np.sum(y3[1:-1][np.arange(n3-1)%3!=2]) + 2*np.sum(y3[3:-1:3]))
    print(f"{n:>5}  {tr:>14.8f}  {s1:>14.8f}  {s3:>14.8f}")


    n        Trapézio        Simp 1/3        Simp 3/8
    6      0.09451810      0.12533187      0.10420688
   12      0.10712688      0.11132981      0.11623592
   24      0.10769047      0.10787834      0.10813667


## Questão 7
Quadratura de Gauss: ∫₀³ x eˣ dx (exato = 2e³+1).

In [8]:
f = lambda x: x*np.exp(x)
exato = 2*np.e**3 + 1
print(f"Exato = {exato:.8f}")
for n in [2, 4, 6]:
    val = gauss_quad(f, 0, 3, n)
    print(f"n={n}: {val:.8f}  erro={abs(val-exato):.2e}")


Exato = 41.17107385
n=2: 39.60750200  erro=1.56e+00
n=4: 41.17056992  erro=5.04e-04
n=6: 41.17107383  erro=1.90e-08


## Questão 8
Distância percorrida pelo avião: ∫₄₀⁹³ 97000v/(5v²+570000) dv.

In [9]:
f = lambda v: 97000*v/(5*v**2 + 570000)
ref = gauss_quad(f, 40, 93, 20)
print(f"Referência (Gauss 20 pontos): {ref:.6f} m")
for n in [10, 50, 200]:
    val = simp13(f, 40, 93, n)
    print(f"Simpson 1/3 n={n:3d}: {val:.6f} m  erro={abs(val-ref):.2e}")


Referência (Gauss 20 pontos): 574.149413 m
Simpson 1/3 n= 10: 574.149431 m  erro=1.79e-05
Simpson 1/3 n= 50: 574.149413 m  erro=2.86e-08
Simpson 1/3 n=200: 574.149413 m  erro=1.12e-10


## Questão 9
Bola de futebol — área e volume por integração numérica tabelada.

In [10]:
z = np.array([0,2.5,5.1,7.6,10.2,12.7,15.2,17.8,20.3,22.9,25.4,27.9,30.5])
d = np.array([0,6.6,8.1,12.2,14.2,15.2,15.7,15.2,14.2,12.2,8.1,6.6,0])
r = d/2; h = z[1]-z[0]  # passo uniforme

def simp13_tab(y, h):
    n = len(y)-1
    return h/3*(y[0]+y[-1] + 4*np.sum(y[1::2]) + 2*np.sum(y[2:-1:2]))

S = 2*np.pi*simp13_tab(r, h)
V = np.pi*simp13_tab(r**2, h)
print(f"Área superficial S = {S:.4f} cm²")
print(f"Volume V           = {V:.4f} cm³")


Área superficial S = 1027.8244 cm²
Volume V           = 3239.4402 cm³


## Questão 10
Comprimento do Arco do Gateway Arch.

In [11]:
fprime = lambda x: -(20.97/30.38)*np.sinh(x/30.38)
f = lambda x: np.sqrt(1 + fprime(x)**2)
a, b = -91.21, 91.21
ref = gauss_quad(f, a, b, 40)
print(f"Referência (Gauss 40 pts): {ref:.6f} m")
for n in [8, 20, 50]:
    print(f"Simpson 1/3 n={n:2d}: {simp13(f,a,b,n):.6f} m  erro={abs(simp13(f,a,b,n)-ref):.2e}")


Referência (Gauss 40 pts): 451.397358 m
Simpson 1/3 n= 8: 452.090612 m  erro=6.93e-01
Simpson 1/3 n=20: 451.416076 m  erro=1.87e-02
Simpson 1/3 n=50: 451.397842 m  erro=4.84e-04
